In [8]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 1200

jarak = np.random.exponential(scale=30, size=n).clip(1, 200).astype(int)
usia = np.random.normal(loc=35, scale=12, size=n).clip(18, 80).astype(int)
hubungan = np.random.choice(
    ['Keluarga', 'Teman_Kuliah', 'Teman_Kantor', 'Teman_SMA', 'Tetangga'],
    size=n, p=[0.25, 0.20, 0.25, 0.15, 0.15]
)
pernah_hadir = np.random.choice([0, 1], size=n, p=[0.6, 0.4])
jumlah_keluarga = np.random.poisson(lam=1.5, size=n).clip(0, 6).astype(int)
punya_balita = np.random.choice([0, 1], size=n, p=[0.7, 0.3])
hari_akad = np.random.choice(
    ['Weekday', 'Weekend', 'Libur_Nasional'],
    size=n, p=[0.4, 0.45, 0.15]
)

# Logika target yang lebih realistis
hadir_akad = []

for i in range(n):
    score = 0
    # Jarak: makin jauh makin kecil kemungkinan hadir
    if jarak[i] < 15: score += 2
    elif jarak[i] < 50: score += 1
    elif jarak[i] > 100: score -= 2
    else: score -= 1

    # Hubungan: keluarga lebih mungkin hadir
    if hubungan[i] == 'Keluarga': score += 3
    elif hubungan[i] == 'Teman_Kantor': score += 1
    elif hubungan[i] == 'Tetangga': score -= 1

    # Pernah hadir sebelumnya
    if pernah_hadir[i] == 1: score += 2

    # Jumlah keluarga diajak: makin banyak makin besar kemungkinan hadir
    if jumlah_keluarga[i] >= 3: score += 1

    # Punya balita: mengurangi kemungkinan hadir
    if punya_balita[i] == 1: score -= 2

    # Hari akad
    if hari_akad[i] == 'Weekend': score += 1
    elif hari_akad[i] == 'Libur_Nasional': score += 2

    # Konversi ke probabilitas
    prob = 1 / (1 + np.exp(-score * 0.4))
    hadir_akad.append(1 if np.random.random() < prob else 0)

df = pd.DataFrame({
    'jarak_ke_venue_km': jarak,
    'usia': usia,
    'hubungan': hubungan,
    'pernah_hadir_sebelumnya': pernah_hadir,
    'jumlah_keluarga_diajak': jumlah_keluarga,
    'punya_balita': punya_balita,
    'hari_akad': hari_akad,
    'hadir_akad': hadir_akad
})

df.to_csv('konfirmasi_akad_nikah.csv', index=False)
print(f"Dataset berhasil dibuat: {len(df)} baris")
print(f"Distribusi target:")
print(df['hadir_akad'].value_counts())
print(df.head())

Dataset berhasil dibuat: 1200 baris
Distribusi target:
hadir_akad
1    875
0    325
Name: count, dtype: int64
   jarak_ke_venue_km  usia      hubungan  pernah_hadir_sebelumnya  \
0                 14    53     Teman_SMA                        0   
1                 90    34     Teman_SMA                        1   
2                 39    28  Teman_Kantor                        0   
3                 27    57      Tetangga                        0   
4                  5    18      Keluarga                        1   

   jumlah_keluarga_diajak  punya_balita       hari_akad  hadir_akad  
0                       1             1         Weekend           1  
1                       2             0         Weekend           1  
2                       1             0         Weekend           1  
3                       0             0         Weekend           1  
4                       1             1  Libur_Nasional           1  


In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils import resample

df = pd.read_csv("konfirmasi_akad_nikah.csv")

df['kategori_usia'] = pd.cut(
    df['usia'], 
    bins=[0, 10, 18, 50, df["usia"].idxmax()], 
    labels=['Anak-anak','Remaja', 'Dewasa', 'Tua']
)

df['kategori_jarak'] = pd.cut(
    df['jarak_ke_venue_km'], 
    bins=[0, 15, 60, df["jarak_ke_venue_km"].idxmax()], 
    labels=['Dekat', 'Menengah', 'Jauh']
)

df['jumlah_keluarga'] = pd.cut(
    df['jumlah_keluarga_diajak'], 
    bins=[-1, 0, 1, 2, df["jumlah_keluarga_diajak"].idxmax()], 
    labels=['Sendiri', 'Berdua', 'Bertiga', 'Rombongan']
)

df = df.drop(['usia', 'jarak_ke_venue_km', 'jumlah_keluarga_diajak'], axis=1)

df_hadir = df[df['hadir_akad'] == 1]
df_tidak_hadir = df[df['hadir_akad'] == 0]

print(f"Jumlah sampel sebelum balancing:")
print(f"  Hadir       : {len(df_hadir)}")
print(f"  Tidak Hadir : {len(df_tidak_hadir)}")

df_hadir_downsampled = resample(
    df_hadir,
    replace=False,
    n_samples=len(df_tidak_hadir),
    random_state=42
)
df_balanced = pd.concat([df_hadir_downsampled, df_tidak_hadir])

print(f"Total sampel setelah balancing: {len(df_balanced)}")
print(f"  Hadir       : {len(df_balanced[df_balanced['hadir_akad'] == 1])}")
print(f"  Tidak Hadir : {len(df_balanced[df_balanced['hadir_akad'] == 0])}")

X = df_balanced.drop('hadir_akad', axis=1)
y = df_balanced['hadir_akad']

X_encoded = pd.get_dummies(X, drop_first=True, dtype=int)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

base_model = LogisticRegression(class_weight='balanced', random_state=42, max_iter=2000)

param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'l1_ratio': [0,1],
    'solver': ['liblinear', 'saga'] 
}

grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    cv=5,
    scoring='f1_macro', 
    n_jobs=-1     
)

grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("\n=== Laporan Optimasi Model ===")
print(f"Parameter Terbaik : {grid_search.best_params_}")

print("\n=== Hasil Evaluasi Model ===")
print(f"Akurasi: {accuracy_score(y_test, y_pred):.2f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Tidak Hadir', 'Hadir']))

Jumlah sampel sebelum balancing:
  Hadir       : 875
  Tidak Hadir : 325
Total sampel setelah balancing: 650
  Hadir       : 325
  Tidak Hadir : 325

=== Laporan Optimasi Model ===
Parameter Terbaik : {'C': 10, 'l1_ratio': 0, 'solver': 'liblinear'}

=== Hasil Evaluasi Model ===
Akurasi: 0.65

Classification Report:
              precision    recall  f1-score   support

 Tidak Hadir       0.60      0.68      0.64        60
       Hadir       0.69      0.61      0.65        70

    accuracy                           0.65       130
   macro avg       0.65      0.65      0.65       130
weighted avg       0.65      0.65      0.65       130



In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils import resample

df = pd.read_csv("konfirmasi_akad_nikah.csv")

df['kategori_usia'] = pd.cut(
    df['usia'], bins=[0, 10, 18, 50, df['usia'].max()], 
    labels=['Anak-anak','Remaja', 'Dewasa', 'Tua']
)
df['kategori_jarak'] = pd.cut(
    df['jarak_ke_venue_km'], bins=[0, 15, 60, df['jarak_ke_venue_km'].max()], 
    labels=['Dekat', 'Menengah', 'Jauh']
)
df['jumlah_keluarga'] = pd.cut(
    df['jumlah_keluarga_diajak'], bins=[-1, 0, 1, 2, df['jumlah_keluarga_diajak'].max()], 
    labels=['Sendiri', 'Berdua', 'Bertiga', 'Rombongan']
)

df = df.drop(['usia', 'jarak_ke_venue_km', 'jumlah_keluarga_diajak'], axis=1)

print(f"Total data: {len(df)}")
print(f"  Hadir       : {len(df[df['hadir_akad'] == 1])}")
print(f"  Tidak Hadir : {len(df[df['hadir_akad'] == 0])}")

df_hadir = df[df['hadir_akad'] == 1]
df_tidak_hadir = df[df['hadir_akad'] == 0]

df_hadir_downsampled = resample(
    df_hadir, replace=False, n_samples=len(df_tidak_hadir), random_state=42
)
df_balanced = pd.concat([df_hadir_downsampled, df_tidak_hadir])

print(f"\nSetelah undersampling: {len(df_balanced)}")
print(f"  Hadir       : {len(df_balanced[df_balanced['hadir_akad'] == 1])}")
print(f"  Tidak Hadir : {len(df_balanced[df_balanced['hadir_akad'] == 0])}")

X = df_balanced.drop('hadir_akad', axis=1)
y = df_balanced['hadir_akad']

X_encoded = pd.get_dummies(X, drop_first=True, dtype=int)

print(f"\nJumlah fitur setelah encoding: {X_encoded.shape[1]}")
print(f"Nama fitur: {list(X_encoded.columns)}")

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

# StandardScaler terpisah
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Logistic Regression dengan class_weight
base_model = LogisticRegression(
    class_weight='balanced', random_state=42, max_iter=2000
)

param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear'],
    'l1_ratio': [0, 1]
}

grid_search = GridSearchCV(
    base_model, param_grid, cv=5, scoring='f1_macro', n_jobs=-1
)
grid_search.fit(X_train_scaled, y_train)

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_scaled)

print("\n=== Laporan Optimasi Model ===")
print(f"Parameter Terbaik : {grid_search.best_params_}")

print("\n=== Hasil Evaluasi Model ===")
print(f"Akurasi: {accuracy_score(y_test, y_pred):.2f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Tidak Hadir', 'Hadir']))

Total data: 1200
  Hadir       : 875
  Tidak Hadir : 325

Setelah undersampling: 650
  Hadir       : 325
  Tidak Hadir : 325

Jumlah fitur setelah encoding: 16
Nama fitur: ['pernah_hadir_sebelumnya', 'punya_balita', 'hubungan_Teman_Kantor', 'hubungan_Teman_Kuliah', 'hubungan_Teman_SMA', 'hubungan_Tetangga', 'hari_akad_Weekday', 'hari_akad_Weekend', 'kategori_usia_Remaja', 'kategori_usia_Dewasa', 'kategori_usia_Tua', 'kategori_jarak_Menengah', 'kategori_jarak_Jauh', 'jumlah_keluarga_Berdua', 'jumlah_keluarga_Bertiga', 'jumlah_keluarga_Rombongan']

=== Laporan Optimasi Model ===
Parameter Terbaik : {'C': 10, 'l1_ratio': 1, 'solver': 'liblinear'}

=== Hasil Evaluasi Model ===
Akurasi: 0.65

Classification Report:
              precision    recall  f1-score   support

 Tidak Hadir       0.60      0.70      0.65        60
       Hadir       0.70      0.60      0.65        70

    accuracy                           0.65       130
   macro avg       0.65      0.65      0.65       130
weighted

In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils import resample

df = pd.read_csv("konfirmasi_akad_nikah.csv")

df['kategori_usia'] = pd.cut(
    df['usia'], bins=[0, 10, 18, 50, df['usia'].max()], 
    labels=['Anak-anak','Remaja', 'Dewasa', 'Tua']
)
df['kategori_jarak'] = pd.cut(
    df['jarak_ke_venue_km'], bins=[0, 15, 60, df['jarak_ke_venue_km'].max()], 
    labels=['Dekat', 'Menengah', 'Jauh']
)
df['jumlah_keluarga'] = pd.cut(
    df['jumlah_keluarga_diajak'], bins=[-1, 0, 1, 2, df['jumlah_keluarga_diajak'].max()], 
    labels=['Sendiri', 'Berdua', 'Bertiga', 'Rombongan']
)

df = df.drop(['usia', 'jarak_ke_venue_km', 'jumlah_keluarga_diajak'], axis=1)

print(f"Total data: {len(df)}")
print(f"  Hadir       : {len(df[df['hadir_akad'] == 1])}")
print(f"  Tidak Hadir : {len(df[df['hadir_akad'] == 0])}")

df_hadir = df[df['hadir_akad'] == 1]
df_tidak_hadir = df[df['hadir_akad'] == 0]

df_hadir_downsampled = resample(
    df_hadir, replace=False, n_samples=len(df_tidak_hadir), random_state=42
)
df_balanced = pd.concat([df_hadir_downsampled, df_tidak_hadir])

print(f"\nSetelah undersampling: {len(df_balanced)}")
print(f"  Hadir       : {len(df_balanced[df_balanced['hadir_akad'] == 1])}")
print(f"  Tidak Hadir : {len(df_balanced[df_balanced['hadir_akad'] == 0])}")

X = df_balanced.drop('hadir_akad', axis=1)
y = df_balanced['hadir_akad']

X_encoded = pd.get_dummies(X, drop_first=True, dtype=int)

print(f"\nJumlah fitur setelah encoding: {X_encoded.shape[1]}")
print(f"Nama fitur: {list(X_encoded.columns)}")

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

# StandardScaler terpisah
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Logistic Regression dengan class_weight
base_model = LogisticRegression(
    random_state=42, max_iter=2000
)

param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear'],
    'l1_ratio': [0, 1]
}

grid_search = GridSearchCV(
    base_model, param_grid, cv=5, scoring='f1_macro', n_jobs=-1
)
grid_search.fit(X_train_scaled, y_train)

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_scaled)

print("\n=== Laporan Optimasi Model ===")
print(f"Parameter Terbaik : {grid_search.best_params_}")

print("\n=== Hasil Evaluasi Model ===")
print(f"Akurasi: {accuracy_score(y_test, y_pred):.2f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Tidak Hadir', 'Hadir']))

Total data: 1200
  Hadir       : 875
  Tidak Hadir : 325

Setelah undersampling: 650
  Hadir       : 325
  Tidak Hadir : 325

Jumlah fitur setelah encoding: 16
Nama fitur: ['pernah_hadir_sebelumnya', 'punya_balita', 'hubungan_Teman_Kantor', 'hubungan_Teman_Kuliah', 'hubungan_Teman_SMA', 'hubungan_Tetangga', 'hari_akad_Weekday', 'hari_akad_Weekend', 'kategori_usia_Remaja', 'kategori_usia_Dewasa', 'kategori_usia_Tua', 'kategori_jarak_Menengah', 'kategori_jarak_Jauh', 'jumlah_keluarga_Berdua', 'jumlah_keluarga_Bertiga', 'jumlah_keluarga_Rombongan']

=== Laporan Optimasi Model ===
Parameter Terbaik : {'C': 10, 'l1_ratio': 0, 'solver': 'liblinear'}

=== Hasil Evaluasi Model ===
Akurasi: 0.72

Classification Report:
              precision    recall  f1-score   support

 Tidak Hadir       0.71      0.75      0.73        65
       Hadir       0.74      0.69      0.71        65

    accuracy                           0.72       130
   macro avg       0.72      0.72      0.72       130
weighted

In [21]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils import resample

df = pd.read_csv("konfirmasi_akad_nikah.csv")

df['kategori_usia'] = pd.cut(
    df['usia'], bins=[0, 10, 18, 50, df['usia'].max()], 
    labels=['Anak-anak','Remaja', 'Dewasa', 'Tua']
)
df['kategori_jarak'] = pd.cut(
    df['jarak_ke_venue_km'], bins=[0, 15, 60, df['jarak_ke_venue_km'].max()], 
    labels=['Dekat', 'Menengah', 'Jauh']
)
df['jumlah_keluarga'] = pd.cut(
    df['jumlah_keluarga_diajak'], bins=[-1, 0, 1, 2, df['jumlah_keluarga_diajak'].max()], 
    labels=['Sendiri', 'Berdua', 'Bertiga', 'Rombongan']
)

df = df.drop(['usia', 'jarak_ke_venue_km', 'jumlah_keluarga_diajak'], axis=1)

print(f"Total data: {len(df)}")
print(f"  Hadir       : {len(df[df['hadir_akad'] == 1])}")
print(f"  Tidak Hadir : {len(df[df['hadir_akad'] == 0])}")

df_hadir = df[df['hadir_akad'] == 1]
df_tidak_hadir = df[df['hadir_akad'] == 0]

df_hadir_downsampled = resample(
    df_hadir, replace=False, n_samples=len(df_tidak_hadir), random_state=42
)
df_balanced = pd.concat([df_hadir_downsampled, df_tidak_hadir])

print(f"\nSetelah undersampling: {len(df_balanced)}")
print(f"  Hadir       : {len(df_balanced[df_balanced['hadir_akad'] == 1])}")
print(f"  Tidak Hadir : {len(df_balanced[df_balanced['hadir_akad'] == 0])}")

X = df_balanced.drop('hadir_akad', axis=1)
y = df_balanced['hadir_akad']

X_encoded = pd.get_dummies(X, drop_first=True, dtype=int)

print(f"\nJumlah fitur setelah encoding: {X_encoded.shape[1]}")
print(f"Nama fitur: {list(X_encoded.columns)}")

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

# StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Random Forest dengan data yang sudah di-scale
base_model = RandomForestClassifier(random_state=42, class_weight='balanced')

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(
    base_model, param_grid, cv=5, scoring='f1_macro', n_jobs=-1
)
grid_search.fit(X_train_scaled, y_train)

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_scaled)

print("\n=== Laporan Optimasi Model ===")
print(f"Parameter Terbaik : {grid_search.best_params_}")

print("\n=== Hasil Evaluasi Model ===")
print(f"Akurasi: {accuracy_score(y_test, y_pred):.2f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Tidak Hadir', 'Hadir']))

Total data: 1200
  Hadir       : 875
  Tidak Hadir : 325

Setelah undersampling: 650
  Hadir       : 325
  Tidak Hadir : 325

Jumlah fitur setelah encoding: 16
Nama fitur: ['pernah_hadir_sebelumnya', 'punya_balita', 'hubungan_Teman_Kantor', 'hubungan_Teman_Kuliah', 'hubungan_Teman_SMA', 'hubungan_Tetangga', 'hari_akad_Weekday', 'hari_akad_Weekend', 'kategori_usia_Remaja', 'kategori_usia_Dewasa', 'kategori_usia_Tua', 'kategori_jarak_Menengah', 'kategori_jarak_Jauh', 'jumlah_keluarga_Berdua', 'jumlah_keluarga_Bertiga', 'jumlah_keluarga_Rombongan']

=== Laporan Optimasi Model ===
Parameter Terbaik : {'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 100}

=== Hasil Evaluasi Model ===
Akurasi: 0.74

Classification Report:
              precision    recall  f1-score   support

 Tidak Hadir       0.74      0.74      0.74        65
       Hadir       0.74      0.74      0.74        65

    accuracy                           0.74       130
   macro avg       0.74